In [ ]:
# Lendo os dados brutos da Camada Bronze armazenados no Garage
df_bronze = spark.read.parquet("s3a://meu-data-lake/live_network_logs/")

# Processamento Silver: Limpeza e Padronização
# 1. Filtramos conexões que não trafegaram bytes reais
# 2. Ignoramos tráfego de broadcast puro da rede local
df_silver = df_bronze.filter((col("packets") > 0) & (col("bytes") > 0)) \
    .filter(col("dst_ip") != "255.255.255.255") \
    .dropDuplicates()

# Gravando os dados refinados no bucket da Camada Silver
df_silver.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy("vlan") \
    .save("s3a://meu-data-lake/silver_network_logs/")

print("🥈 Dados limpos e processados na Camada Silver com sucesso!")
# spark.read.parquet("s3a://meu-data-lake/silver_network_logs/").show(5)

In [ ]:
from pyspark.sql.functions import sum as _sum, count, countDistinct, round, when

# Lendo a base de dados confiável da Camada Silver
df_clean = spark.read.parquet("s3a://meu-data-lake/silver_network_logs/")

# Engenharia de Atributos (Data Mining): Agrupando por IP e VLAN
df_gold = df_clean.groupBy("src_ip", "vlan").agg(
    _sum("packets").alias("total_packets"),
    _sum("bytes").alias("total_bytes"),
    
    # Quantos IPs e Portas diferentes este dispositivo tentou acessar? (Entropia simplificada)
    countDistinct("dst_ip").alias("unique_targets"),
    countDistinct("dst_port").alias("unique_ports") 
).withColumn(
    # Calculando a razão Bytes por Pacote (Pacotes pequenos e simétricos indicam força bruta/Mirai)
    "bytes_per_packet", round(col("total_bytes") / col("total_packets"), 2)
)

# Aplicando uma regra de mineração de dados primária (Heurística de Detecção)
# Exemplo: Se um IP atinge muitas portas únicas com poucos bytes por pacote -> Alta probabilidade de Scanner/Botnet
df_gold_analise = df_gold.withColumn(
    "comportamento", 
    when((col("unique_ports") > 10) & (col("bytes_per_packet") < 100), "ANOMALIA_BOTNET")
    .otherwise("NORMAL")
)

# Gravando o dataset analítico final na Camada Gold
df_gold_analise.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("s3a://meu-data-lake/gold_ml_features/")

print("🥇 Camada Gold gerada! Dataset pronto para Machine Learning.")
df_gold_analise.show(truncate=False)